# AI Text Generation for Precog NLP Project
## Google Colab Version with Sequential Generation

**Goal:** Generate 1000 AI text samples (500 Class 2 vanilla, 500 Class 3 author-mimic) using Gemini API.

**Why Colab:**
- No local rate limits or connection issues
- Can run in background
- Automatic save to Google Drive
- Free GPU if needed for future analysis

**Runtime:** ~2-3 hours for 1000 samples with proper throttling

## Setup: Install Dependencies

In [ ]:
# Install required packages
!pip install google-generativeai tqdm pandas -q
print("✅ Packages installed!")

## Mount Google Drive (Optional - for auto-save)

In [ ]:
from google.colab import drive
import os

# Mount Google Drive
drive.mount('/content/drive')

# Create output directory
OUTPUT_DIR = '/content/drive/MyDrive/precog_data'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Change to output directory
os.chdir(OUTPUT_DIR)
print(f"✅ Working directory: {os.getcwd()}")
print(f"   Files will be saved to Google Drive")

## Configuration: API Key

In [ ]:
import google.generativeai as genai
from google.colab import userdata

# Get API key from Colab Secrets
# To set up: Click the key icon (🔑) in the left sidebar > Add new secret
# Name: GEMINI_API_KEY
# Value: Your API key
API_KEY = userdata.get('GEMINI_API_KEY')
genai.configure(api_key=API_KEY)

# Use Gemini Pro Latest
model = genai.GenerativeModel('gemini-pro-latest')

print("✅ Gemini API configured")
print(f"   Model: gemini-pro-latest")
print(f"   API Key: {API_KEY[:10]}...{API_KEY[-4:]} (secured)")

## Entropy Injector: Prevents Mode Collapse

In [ ]:
import random

class EntropyInjector:
    """Randomizes prompt parameters to prevent AI mode collapse"""
    
    def __init__(self):
        # Tones: Broad emotional colors
        self.tones = [
            "Clinical and detached", "Melancholic", "Urgent and fast-paced", 
            "Reflective and nostalgic", "Cynical", "Optimistic and bright",
            "Mysterious", "Formal and academic"
        ]
        
        # Perspectives: Narrative lens
        self.perspectives = [
            "Third-person limited", "Third-person omniscient", 
            "First-person observer" 
        ]
        
        # Constraints: Force unique sentence structures
        self.constraints = [
            "Use a metaphor involving nature.",
            "Start the paragraph with a rhetorical question.",
            "Do not use the word 'very' or 'really'.",
            "Use at least one semicolon to connect independent clauses.",
            "Focus heavily on sensory details (sight, sound, smell).",
            "Keep sentences relatively short and punchy.",
            "Use complex, flowing sentence structures with multiple clauses."
        ]

    def get_prompt_metadata(self, class_type):
        """Return randomized parameters based on class type"""
        if class_type == "class_2":
            # Class 2: Minimal entropy (stays AI-like)
            return {
                "tone": random.choice(["Neutral", "Informative", "Descriptive"]), 
                "constraint": "None", 
                "perspective": "Third-person" 
            }
        
        elif class_type == "class_3":
            # Class 3: High entropy (forces style mimicry)
            return {
                "tone": random.choice(self.tones),
                "constraint": random.choice(self.constraints),
                "perspective": random.choice(self.perspectives)
            }

print("✅ EntropyInjector class loaded")
print("   Class 2: 3 tones × 1 perspective = 3 variations")
print("   Class 3: 8 tones × 3 perspectives × 7 constraints = 168 variations")

## Generation Function: Robust & Resumable

In [ ]:
import pandas as pd
import time
from tqdm import tqdm

def generate_dataset(topics, author_name, class_type, n_samples=500):
    """
    Generate AI text samples with entropy injection.
    
    Features:
    - Resumes from where it left off
    - Saves after every successful generation
    - Handles rate limits with exponential backoff
    - Shows progress bar
    
    Args:
        topics: List of topic strings
        author_name: Author to mimic (for class_3)
        class_type: "class_2" (vanilla) or "class_3" (mimic)
        n_samples: Total samples to generate
    
    Returns:
        DataFrame with generated samples
    """
    
    injector = EntropyInjector()
    output_file = f"{class_type}_vanilla.csv" if class_type == "class_2" else f"{class_type}_mimic.csv"
    
    # 1. Resume Logic
    results = []
    start_idx = 0
    
    if os.path.exists(output_file):
        try:
            existing_df = pd.read_csv(output_file)
            if len(existing_df) > 0:
                results = existing_df.to_dict('records')
                start_idx = len(results)
                print(f"📂 Resuming {class_type} from sample {start_idx}...")
            else:
                print(f"📂 Found empty file. Starting fresh...")
        except (pd.errors.EmptyDataError, pd.errors.ParserError):
            print(f"⚠️ File corrupted. Starting fresh...")
    else:
        print(f"🆕 Starting fresh for {class_type}...")

    # Stop if already complete
    if start_idx >= n_samples:
        print(f"✅ {class_type} already complete ({start_idx} samples).")
        return pd.DataFrame(results)

    print(f"🚀 Generating {n_samples - start_idx} remaining samples...\n")

    try:
        # 2. Generation Loop
        for i in tqdm(range(start_idx, n_samples), initial=start_idx, total=n_samples, desc=class_type):
            # Rotate through topics evenly
            current_topic = topics[i % len(topics)]
            
            # Get randomized parameters
            meta = injector.get_prompt_metadata(class_type)
            
            # Construct prompt
            if class_type == "class_2":
                base_instruction = f"Write a paragraph about '{current_topic}'."
                style_instruction = f"Tone: {meta['tone']}. Keep it between 100-200 words."
            elif class_type == "class_3":
                base_instruction = f"Write a paragraph about '{current_topic}' in the unique style of {author_name}."
                style_instruction = (
                    f"Tone: {meta['tone']}. "
                    f"Perspective: {meta['perspective']}. "
                    f"Constraint: {meta['constraint']} "
                    f"Length: 100-200 words."
                )

            final_prompt = f"{base_instruction}\nInstructions: {style_instruction}\nOutput only the text."

            # 3. API Call with Retry Logic
            max_retries = 5
            for attempt in range(max_retries):
                try:
                    response = model.generate_content(final_prompt)
                    generated_text = response.text.strip()
                    
                    # Log data
                    results.append({
                        "class_label": class_type,
                        "topic": current_topic,
                        "author_target": author_name if class_type == "class_3" else "AI_Generic",
                        "text": generated_text,
                        "prompt_used": final_prompt,
                        "meta_tone": meta['tone'],
                        "meta_constraint": meta['constraint']
                    })
                    
                    # Incremental save
                    pd.DataFrame(results).to_csv(output_file, index=False)
                    
                    # Throttle: 5 seconds = 12 requests/min (safe for free tier)
                    time.sleep(5)
                    break  # Success, move to next sample
                    
                except Exception as e:
                    # Rate limit handling
                    if "429" in str(e) or "quota" in str(e).lower():
                        wait_time = 30 * (attempt + 1)  # 30s, 60s, 90s...
                        tqdm.write(f"⚠️ Rate limit at sample {i}. Waiting {wait_time}s...")
                        time.sleep(wait_time)
                    else:
                        tqdm.write(f"❌ Error at sample {i}: {str(e)[:100]}")
                        time.sleep(10)
                        if attempt == max_retries - 1:
                            tqdm.write(f"⏭️ Skipping sample {i} after {max_retries} failures")

    except KeyboardInterrupt:
        print(f"\n🛑 Interrupted. Progress saved to {output_file}")
        return pd.DataFrame(results)

    print(f"\n✅ {class_type} Complete! Saved to {output_file}")
    print(f"   Total samples: {len(results)}")
    return pd.DataFrame(results)

print("✅ Generation function loaded")

## Define Topics and Author

In [ ]:
# Topics from LDA analysis (Cell 3 from original notebook)
my_topics = [
    "The Science of Deduction",
    "The Curse of Ancestry",
    "Vengeance from the Past",
    "Domestic Tragedy",
    "The Deception of Appearances"
]

my_author = "Arthur Conan Doyle"

print(f"✅ Configuration:")
print(f"   Topics: {len(my_topics)}")
print(f"   Author: {my_author}")
print(f"   Target: 500 samples per class (1000 total)")

## Generate Class 2 (Vanilla AI)

**Estimated time:** ~45-60 minutes for 500 samples

In [ ]:
print("="*60)
print("STARTING CLASS 2: VANILLA AI GENERATION")
print("="*60)

df_class_2 = generate_dataset(
    topics=my_topics,
    author_name=my_author,
    class_type="class_2",
    n_samples=500
)

print(f"\n📊 Class 2 Summary:")
print(df_class_2.head())
print(f"\n   Shape: {df_class_2.shape}")
print(f"   Tone distribution: {df_class_2['meta_tone'].value_counts().to_dict()}")

## Generate Class 3 (Author Mimic)

**Estimated time:** ~45-60 minutes for 500 samples

In [ ]:
print("="*60)
print("STARTING CLASS 3: AUTHOR MIMIC GENERATION")
print("="*60)

df_class_3 = generate_dataset(
    topics=my_topics,
    author_name=my_author,
    class_type="class_3",
    n_samples=500
)

print(f"\n📊 Class 3 Summary:")
print(df_class_3.head())
print(f"\n   Shape: {df_class_3.shape}")
print(f"   Tone distribution: {df_class_3['meta_tone'].value_counts().to_dict()}")
print(f"   Perspective distribution: {df_class_3['meta_constraint'].value_counts().to_dict()}")

## Final Summary and Download

In [ ]:
print("="*60)
print("🎉 GENERATION COMPLETE!")
print("="*60)

# Load final data
df_c2_final = pd.read_csv('class_2_vanilla.csv')
df_c3_final = pd.read_csv('class_3_mimic.csv')

print(f"\n📊 Final Dataset:")
print(f"   Class 2 (Vanilla): {len(df_c2_final)} samples")
print(f"   Class 3 (Mimic): {len(df_c3_final)} samples")
print(f"   Total: {len(df_c2_final) + len(df_c3_final)} samples")

print(f"\n💾 Files saved to:")
print(f"   {os.path.abspath('class_2_vanilla.csv')}")
print(f"   {os.path.abspath('class_3_mimic.csv')}")

if '/content/drive' in os.getcwd():
    print(f"\n✅ Files are saved to Google Drive and will persist!")

print(f"\n🚀 Next steps:")
print(f"   1. Download CSVs from Google Drive")
print(f"   2. Train classifier on this data")
print(f"   3. Analyze stylometric features")

## Optional: Download Files Directly

Run this to download CSVs to your local machine from Colab.

In [ ]:
from google.colab import files

# Download generated files
files.download('class_2_vanilla.csv')
files.download('class_3_mimic.csv')

print("✅ Files downloaded to your computer!")

---

## Troubleshooting

**If you hit rate limits:**
- Increase sleep time in generation function (line with `time.sleep(5)`)
- The function will auto-retry with exponential backoff
- Progress is saved after each sample, so you can stop and resume

**If Colab disconnects:**
- Reconnect and re-run from the generation cell
- The function will resume from where it left off
- Data is auto-saved to Google Drive

**To check progress:**
```python
import pandas as pd
print(f"Class 2: {len(pd.read_csv('class_2_vanilla.csv'))} samples")
print(f"Class 3: {len(pd.read_csv('class_3_mimic.csv'))} samples")
```